In [30]:
!pip install -q crewai
!pip install -q openai
!pip install -q unstructured
!pip install -q tools
!pip install -q tenacity==8.3.0
!pip install -q langchain
!pip install -q langchain_groq
!pip install -q cohere
!pip install -q langchain_community
!pip install -q 'crewai[tools]'

In [31]:
from crewai import Agent, Task, Crew
from langchain_community.chat_models import ChatCohere
from langchain_openai import OpenAI
from langchain_groq import ChatGroq


In [32]:
#warning control
import warnings
warnings.filterwarnings('ignore')

In [33]:
import os

COHERE_API_KEY="ieLPQ8jIDR7czKpyAcRYPJTdjjP27AAVwR7Gvwzs"
OPENAI_API_KEY="sk-crew-ai-gqQfhuNQaIn8AKoIRg8UT3BlbkFJfEam04Z3SQEPwjWabC4Y"
GROQ_API_KEY="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"
SERPER_API_KEY= "25b084c4ee79d46c1866395746dbdf145c1729d9"


os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY
os.environ['COHERE_API_KEY'] = COHERE_API_KEY
os.environ['GROQ_API_KEY'] = GROQ_API_KEY
os.environ['SERPER_API_KEY'] = SERPER_API_KEY

#Tools
from crewai_tools import (
    SerperDevTool,
    WebsiteSearchTool
)

search_tool = SerperDevTool()
web_search_tool = WebsiteSearchTool()

#LLMs

cohere = ChatCohere(cohere_api_key=COHERE_API_KEY,
                    temperaature= 0.3)
openai = OpenAI(api_key=OPENAI_API_KEY)
groq = ChatGroq(
                temperature=0,
                groq_api_key=GROQ_API_KEY,
                model_name="mixtral-8x7b-32768"
            )


In [34]:
#Agent
idealCustomer_profiler = Agent(
                         role='Customer Insight Analyst',
                         goal="""Identify and provide an ideal customer profile and companies based on the specified
                                 {industry} industry by the user. The ICP should include detailed insights that
                                 help in targeting and understanding the best potential customers for the {service} service
                                 being offered by the user. This should include detailed insights that help in targeting and understanding
                                 the best potential customers for a product or service within the industry.""",

                         backstory=""" You are a renowned customer insight analyst firm Nielsen with extensive knowledge
                                        in audience insights, data and analytics,and shapes the future of businesses with accurate measurement of what people listen, buy or
                                        show interest within. You also understand that an ideal customer profile is a detailed description of the
                                        type of customer and their companies within the {industry} industry, most likely to benefit from
                                         and be interested in the {service} provided by the user. Your role is to make the process of
                                         identifying and targeting these profiles more efficient by focusting on industry-specific characteristics.""",

                        allow_delegation=False,
                        verbose=True,
                        llm=cohere,
                        output_file="idealCustomerProfile.txt",
                        #tools=[search_tool]
                        #tools=[web_search_tool]
                       )

In [35]:
idealCustomerProfile = Task (
       description= """
                    Here are a detailed overview of the tasks to perform:

                    1. Collect comprehensive data on the companies within the {industry} industry. This data should include,
                       the industry customer characteristics,common pain points, and {industry} industry trends.
                       Utilize sources like industry reports, market research,
                       customer surveys, and case studies.

                    2. Develop ICP templates for each company identified within the {industry} industry that includes:
                        - Demographics: Company name, Typical company size, revenue, job titles.
                        - Pain points: Common challenges faced by businesses in the industry.
                        - Behavioural Traits: Decision-making processes and buying behaviors.
                        - Company Characteristics: Typical growth stage and technology adoption.
                        - Budget: General budget ranges for relevant to {service} and related products.
                   """,
            expected_output="""
                            Generate and provide five detailed ideal customer profile that includes:
                            - Demographics: Company name, Typical company size, revenue, job titles.
                            - Pain points: Common challenges faced by businesses in the industry.
                            - Behavioural Traits: Decision-making processes and buying behaviors.
                            - Company Characteristics: Typical growth stage and technology adoption.
                            - Budget: General budget ranges for relevant to {service} and related products.
                            """,
            agent=idealCustomer_profiler)

In [36]:
#Crew
crew = Crew(
    agents=[idealCustomer_profiler],
    tasks=[idealCustomerProfile],
    verbose=True,
)

In [37]:
#Execute Crew
result = crew.kickoff(inputs={
    "service": "Digital marketing",
    "industry": "NGO"

     })

 [2024-09-09 07:46:23][DEBUG]: == Working Agent: Customer Insight Analyst
 [2024-09-09 07:46:23][INFO]: == Starting Task: 
                    Here are a detailed overview of the tasks to perform:

                    1. Collect comprehensive data on the companies within the NGO industry. This data should include, 
                       the industry customer characteristics,common pain points, and NGO industry trends. 
                       Utilize sources like industry reports, market research,
                       customer surveys, and case studies.

                    2. Develop ICP templates for each company identified within the NGO industry that includes:
                        - Demographics: Company name, Typical company size, revenue, job titles.
                        - Pain points: Common challenges faced by businesses in the industry.
                        - Behavioural Traits: Decision-making processes and buying behaviors.
                        - Company Charac